In [34]:
import os 
import subprocess

In [35]:
import json
import pandas as pd
df = pd.read_csv("latin_square.csv")
index = 0
p1 = df.iloc[0]


#研究被験者の取得
participant = f"P{index+1}"
taskSet = p1["Task Set 1"]
condition_num = int(p1["Condition 1_ConditionNum"])




#ArrangeDataを取得する
with open(f"../InteractiveSmartHome/Assets/EXPERIMENT/ArrangeData/PreTaskArrangement{taskSet}.json", "r", encoding="utf-8") as f:
    arrangeData = json.loads(f.read())

with open(f"../InteractiveSmartHome/Assets/EXPERIMENT/VOICE_LOG/{participant}/{condition_num}_{taskSet}.json", "r", encoding="utf-8") as f:
    voice_log = json.loads(f.read())
    


In [36]:
with open(f"../LLMServer/ExperimentData/RESULTS/P1/P1_1.json", "r", encoding="utf-8") as f:
    result_data = json.loads(f.read())


result_data[-1]

{'user_prompt': '今見ている天井のライト。 左の列を赤 真ん中の列を黄色。 右の列を青にしてください 合計六個のライトが 転倒するはずです',
 'filterAgent': {'output_tool_selection': {'filter_type': 'fov',
   'params': {'isInFov': True, 'order': 'centrality', 'range': 0.0},
   'reasoning': '『左の列』『真ん中の列』『右の列』という視覚的レイアウトの表現があり、身体参照ではないため、fovを選んだ。'},
  'devices': [{'id': '2dc8ca39-d023-42a5-874d-76fb312b5f00',
    'name': 'Ceiling Light 5',
    'position': {'x': -0.4464956, 'y': 2.23415518, 'z': 4.37419748},
    'distance_from_user': 4.931978,
    'eye_centrality_score': 2.91055417},
   {'id': '58126407-cde0-4d62-a13d-1a24bf8ea5f3',
    'name': 'Ceiling Light 6',
    'position': {'x': 0.3735784, 'y': 2.23415518, 'z': 2.55757856},
    'distance_from_user': 3.416463,
    'eye_centrality_score': 15.4298306},
   {'id': '2125de35-62c2-40db-8bee-f0580a92ee8b',
    'name': 'Ceiling Light 1',
    'position': {'x': 1.34820163, 'y': 2.222155, 'z': 5.21053457},
    'distance_from_user': 5.822825,
    'eye_centrality_score': 18.6302319},
   {'id': 'c689031b-92

In [38]:
for log in voice_log:
    print(log)

{'taskId': '7612d934-759d-4a55-9a58-ef65940b7162', 'taskAttemptCount': 0, 'finalId': '516c5e1e-68d5-443b-ba5c-e8c3c3d9c750', 'taskAttempts': [{'attemptId': '516c5e1e-68d5-443b-ba5c-e8c3c3d9c750', 'taskElapsedTime': '28.12045', 'userCommand': '', 'outputDevices': ['2125de35-62c2-40db-8bee-f0580a92ee8b', '7b870bf8-74c9-4aac-9e1e-b0e209a6472f', '4cf87da5-fadb-4383-b959-8562091cb367', 'dcc0f42b-a4c0-4cdb-8d11-cc7e3e30ed71', '84133dee-2ea3-4fcb-9036-56f264096fac']}]}
{'taskId': 'b0e9f035-c92f-4dc6-bd0a-a10b2941f902', 'taskAttemptCount': 0, 'finalId': '98566e1f-93be-4faa-a811-f591a4c09373', 'taskAttempts': [{'attemptId': '98566e1f-93be-4faa-a811-f591a4c09373', 'taskElapsedTime': '20.92029', 'userCommand': '', 'outputDevices': ['4cf87da5-fadb-4383-b959-8562091cb367']}]}
{'taskId': 'ff4748e2-cd97-45bb-bf2c-e5c9c33f78ac', 'taskAttemptCount': 1, 'finalId': 'a405c211-98cb-4a9a-8af0-c772351b90d0', 'taskAttempts': [{'attemptId': 'a405c211-98cb-4a9a-8af0-c772351b90d0', 'taskElapsedTime': '26.26041',

In [29]:
voice_log[4]

{'taskId': '1ff814e6-7f85-4e81-ad9f-9aababb20e6d',
 'taskAttemptCount': 6,
 'finalId': 'f249f0a0-a011-44af-acf3-5bac0f96b87c',
 'taskAttempts': [{'attemptId': 'c38d5025-9b2c-4b1e-ae6b-05b1cecb8e46',
   'taskElapsedTime': '18.76024',
   'userCommand': '',
   'outputDevices': ['7b870bf8-74c9-4aac-9e1e-b0e209a6472f',
    '6dc0f2a7-49f5-4feb-acb5-c243e65c6241',
    '6093cfc6-9ec0-4c71-948e-6e82e920bdb6',
    '84133dee-2ea3-4fcb-9036-56f264096fac']},
  {'attemptId': 'e15febe4-c33a-487e-976d-a6e55b7c4290',
   'taskElapsedTime': '16.58019',
   'userCommand': '',
   'outputDevices': []},
  {'attemptId': 'deb4b24b-adab-4e4c-a852-50c5db5265bb',
   'taskElapsedTime': '0',
   'userCommand': '',
   'outputDevices': []},
  {'attemptId': '55e50d94-659b-4ca0-ae2f-105298617313',
   'taskElapsedTime': '0',
   'userCommand': '',
   'outputDevices': ['4cf87da5-fadb-4383-b959-8562091cb367']},
  {'attemptId': 'a192d9a1-bfa5-475f-9d01-40954aedba17',
   'taskElapsedTime': '28.16045',
   'userCommand': '',
   

In [9]:
def evaluate_all_predictions(output_data_list, arrange_data_list):
    results = []

    # arrange_data を dict に変換して素早くアクセス
    arrange_map = {
        entry["device_arrange_id"]: [dev["deviceId"] for dev in entry["devices"]]
        for entry in arrange_data_list
    }

    for output in output_data_list:
        task_id = output.get("task_id")
        predicted_ids = [d["id"] for d in output.get("selected_devices", [])]
        ground_truth_ids = arrange_map.get(task_id)

        if ground_truth_ids is None:
            print(f"[⚠️ Warning] Ground truth not found for task_id: {task_id}")
            continue

        gt_set = set(ground_truth_ids)
        pred_set = set(predicted_ids)

        true_positives = gt_set & pred_set
        false_positives = pred_set - gt_set
        false_negatives = gt_set - pred_set

        precision = len(true_positives) / len(pred_set) if pred_set else 0
        recall = len(true_positives) / len(gt_set) if gt_set else 0

        results.append({
            "task_id": task_id,
            "ground_truth": ground_truth_ids,
            "predicted": predicted_ids,
            "true_positives": list(true_positives),
            "false_positives": list(false_positives),
            "false_negatives": list(false_negatives),
            "precision": precision,
            "recall": recall
        })

    return results


In [31]:
results = evaluate_all_predictions(outputData, arrangeData)
import pprint
pprint.pprint(results)


NameError: name 'outputData' is not defined